# US Tornado Data: Getting Started

**2010–2025 · release v2.2.1 · Kaggle dataset version 8**

Start with two small ML views (about **3.9 MB** combined), each preserving all **20,164 tornadoes**. Explore **17 linked analysis tables** when you need source records, geometry or provenance. The full reproducible payload is about **1.34 GB**; this notebook downloads only its selected files.

Sources: SPC tornado tracks; NCEI Storm Events, fatalities and locations; NOAA Event Footprint Catalog damage regions; Census county population/housing and generalized county boundaries; NEXRAD Level III indicators through NOAA SWDI; NWS warning updates through IEM; and USGS Annual NLCD land cover. ERA5 and ACS/TIGER tract enrichment are deferred and excluded.

EF ratings classify surveyed damage, not directly measured peak wind. Unknown ratings remain missing. Detailed damage footprints can be surveyed DAT geometry or Storm Events reconstructions, not independent storm wind measurements.

## Load and verify the pinned release

Attach **dataset version 8** before saving a Kaggle run. `kagglehub` uses the attached dataset cache on Kaggle and downloads individual files elsewhere. No repository modules or hard-coded Kaggle paths are required. Locally install `kagglehub pandas pyarrow matplotlib`.

The release manifest has a pinned SHA-256; every selected file is checked against it. The optional `TORNADO_DATA_ROOT` environment variable supports testing against an already-downloaded release. Hugging Face users can use `snapshot_download(..., repo_type="dataset", revision="v2.2.1")` instead; both hosts contain identical shared payload files.

In [ ]:
from pathlib import Path
import hashlib
import json
import os

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
DATASET = "jakevanslyke/us-tornado-data-2010-2025/versions/8"
RELEASE = "v2.2.1"
MANIFEST_SHA256 = "9a4ec9b89cbf43cdfa88db26360881aab84437fee619f1b98e1fb04a338f8feb"
local_root = os.environ.get("TORNADO_DATA_ROOT")

def get_file(relative):
    return (Path(local_root) / relative if local_root else
            Path(kagglehub.dataset_download(DATASET, path=relative)))

def sha256(path):
    return hashlib.file_digest(path.open("rb"), "sha256").hexdigest()

manifest_path = get_file("release_manifest.json")
assert sha256(manifest_path) == MANIFEST_SHA256
manifest = json.loads(manifest_path.read_text())
assert manifest["version"] == RELEASE
expected = {entry["path"]: entry for entry in manifest["files"]}
verified_paths = set()

def verified_file(relative):
    path = get_file(relative)
    assert path.stat().st_size == expected[relative]["bytes"], relative
    assert sha256(path) == expected[relative]["sha256"], relative
    verified_paths.add(relative)
    return path

onset = pd.read_parquet(verified_file("ml/events_onset.parquet"))
retrospective = pd.read_parquet(verified_file("ml/events_retrospective.parquet"))
dictionary = json.loads(verified_file("ml/feature_dictionary.json").read_text())
spc = pd.read_parquet(verified_file("analysis/tornadoes.parquet"))
coverage = pd.read_parquet(verified_file("analysis/source_coverage.parquet"))
print(f"Loaded {len(onset):,} tornadoes; {onset.shape[1]} onset and {retrospective.shape[1]} retrospective columns.")

## All linked tables

The inventory below comes from the verified schema; it does not load every large table. `tornado_id` links the event views and event/source bridges. NCEI `event_id` links its own details, fatalities and locations; use the accepted `source_crosswalk` links to connect NCEI and footprint records to tornadoes. Radar and warning records have their own IDs, linked through `tornado_radar` and `tornado_warnings`.

Keep one-to-many links explicit: a direct join of every detection to every warning can multiply rows. `record_provenance` and `source_coverage` preserve source provenance and collection outcomes.

In [ ]:
schema = json.loads(verified_file("schema.json").read_text())
rows = []
for key, directory in [("analysis_tables", "analysis"),
                       ("additional_analysis_tables", "analysis"),
                       ("ml_tables", "ml")]:
    for entry in schema[key]:
        if entry["path"].endswith(".parquet"):
            columns = entry["columns"]
            rows.append({"table": f"{directory}/{entry['path']}",
                         "rows": entry["rows"],
                         "columns": columns if isinstance(columns, int) else len(columns),
                         "MB": round(entry["bytes"] / 1e6, 2)})
inventory = pd.DataFrame(rows)
assert len(inventory) == 19
assert inventory["table"].str.startswith("analysis/").sum() == 17
display(inventory)
# To inspect a larger source table later:
# radar = pd.read_parquet(verified_file("analysis/radar_detections.parquet"))

## Catalog, EF labels and collection coverage

The catalog preserves every SPC tornado row, including **1,525 unknown ratings**. Collection completed for radar and warnings for every event and NLCD for **20,147** events; **17** are outside the supported NLCD footprint. A completed query does not guarantee a detection, warning or usable value. Missing radar detections do not prove absent radar coverage.

Coverage and reporting practices vary by year and location; do not silently drop events missing an enrichment source.

In [ ]:
assert len(spc) == len(onset) == len(retrospective) == 20164
assert onset["tornado_id"].is_unique and retrospective["tornado_id"].is_unique
for frame in [onset, retrospective]:
    targets = frame.set_index("tornado_id")["target_ef_rating"].sort_index()
    original = spc.set_index("tornado_id")["ef_rating"].sort_index()
    pd.testing.assert_series_equal(targets, original, check_dtype=False, check_names=False, check_index_type=False)
assert onset["target_ef_rating"].isna().sum() == 1525
assert len(coverage) == 60492
display(coverage.groupby(["source", "status"]).size().rename("event_jobs").to_frame())
ratings = onset["target_ef_rating"].value_counts().reindex(range(6), fill_value=0)
ratings.index = [f"EF{i}" for i in ratings.index]
ratings.loc["Unknown"] = onset["target_ef_rating"].isna().sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
spc.groupby("year").size().plot.bar(ax=axes[0], color="#277e8e", title="Recorded tornadoes by year")
ratings.plot.bar(ax=axes[1], color="#d88c3a", title="Final EF ratings")
axes[0].set_ylabel("Tornadoes")
axes[1].set_ylabel("Tornadoes")
plt.tight_layout()
plt.show()
research = json.loads(verified_file("research/summary.json").read_text())
assert research["events"] == 20164
assert research["metric_counts"]["post_land_developed_fraction_missing"] == 317
display(pd.Series(research["definitions"], name="Coverage interpretation").to_frame())


## Build an onset feature matrix

The prediction cutoff is **reported tornado onset**. The dictionary explicitly identifies candidate predictors; targets, IDs, grouping fields and post-event fields are excluded. This is a hindsight event-anchored experiment: reported onset/location come from the final catalog, and radar availability assumes a **five-minute latency**, not a measured dissemination timestamp.

SWDI supplies TVS, mesocyclone and storm-structure detections, associated within 20 km and ±60 minutes. Onset aggregates use only the pre-cutoff subset; proximity does not prove that a detection belongs to the tornado. IEM contains tornado/severe-thunderstorm warning updates. Standalone cancellation messages are absent from this archive path, so active counts/lead times are reconstructed operational approximations.

Use event/outbreak groups and temporal holdouts for evaluation; never split related source rows independently. Impute and fit transformations using training data only. This example prepares the matrix without fitting a model.

Dictionary schema 2 separates **conditional onset eligibility** from **verified availability**. The latter is unknown: final reported time/location are hindsight anchors and radar latency is assumed. Counts/maxima are independently verified against recorded timestamps, but actual receipt and complete cancellations are not established. Coverage reports in `research/` break down missingness by year, state/territory and EF class.

In [ ]:
predictors = dictionary["onset_predictor_columns"]
assert dictionary["schema_version"] == 2
assert all(dictionary["columns"][name]["eligible_for_conditional_onset"] for name in predictors)
assert all(dictionary["columns"][name]["available_by_onset"] is None for name in predictors)
assert predictors and not any(name.startswith("post_") for name in predictors)
known = onset["target_ef_rating"].notna()
X = onset.loc[known, predictors].copy()
y = onset.loc[known, "target_ef_rating"].astype("int64")
groups = onset.loc[known, "ml_split_group"].copy()
print(f"X: {X.shape}; labels: {len(y):,}; unknown labels retained in source views: {(~known).sum():,}")
display(pd.DataFrame({"feature": predictors,
                      "missing_fraction": X.isna().mean().to_numpy()}))
fig, ax = plt.subplots(figsize=(10, 4))
X.isna().mean().sort_values().plot.barh(ax=ax, color="#277e8e")
ax.set_xlabel("Missing fraction among labeled tornadoes")
plt.tight_layout()
plt.show()

## Retrospective paths, land cover and exposure context

The retrospective view adds final duration/path dimensions, post-event radar aggregates and land-cover summaries. NLCD uses native 30 m land-cover and imperviousness pixels from the prior year, summarized over accepted damage footprints or a 500 m fallback track buffer. These path-dependent fields are post-event information and are excluded from onset predictors.

County population/housing are broad context, not people or buildings actually struck. The generalized 2020 county map is not a fine exposure layer. Final damage, casualties, footprint geometry and narratives can leak the EF target or reflect assessment practices. Review these fields individually before any retrospective model.

No environmental reanalysis, tract-level ACS/TIGER enrichment, raw radar volumes or survey photos are included.

In [ ]:
post_columns = [name for name in retrospective if name.startswith("post_")]
display(retrospective[["tornado_id", "target_ef_rating"] + post_columns].head())
assert retrospective.shape == (20164, 53) and onset.shape == (20164, 30)
assert len(verified_paths) == 7
print("Inspection passed: v2.2.1 / Kaggle 8; 17 linked analysis tables and 2 ML views.")
print(f"Pinned release manifest and {len(verified_paths)} selected files passed SHA-256 verification.")
print("All 20,164 event IDs and EF targets preserved in both ML views.")

Source documentation, full schemas, provenance, licenses and reproducible collection/feature code:
[dataset repository](https://github.com/jakeryderv/us-tornado-data-2010-2025),
[shared dataset card](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/main/docs/DATASET_CARD.md),
[enrichment methods](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/main/docs/ENRICHMENT.md),
[release receipt](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/main/release/v2.2.1.json).

The full original-source payload is available in Kaggle's `release.zip.bin`; use the documented trusted checksums when unpacking. Individual analysis/ML downloads are sufficient to begin modeling.